In [ ]:
### Import Setup ##

import sys
import os

#####################
# Import of utils.py functions
#####################
# Required to get utils.py and access its functions
code_dir = os.path.abspath("")
from utils import loadFSL, FSLeyesServer, mkdir_no_exist
repository_dir = os.path.abspath(os.path.join(code_dir, '..'))



#############################
# Loading fsl and freesurfer within Neurodesk
#############################
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.load('freesurfer/7.4.1')
await lmod.list()

####################
# Setup FSL path
####################
loadFSL()

###################
# Load all relevant libraries for the lab
##################
import fsl.wrappers
from fsl.wrappers import fslmaths

import os.path as op
import glob

# FSL function wrappers which we will call from python directly
from fsl.wrappers import fast, bet
from fsl.wrappers.misc import fslroi
from fsl.wrappers import flirt

# General purpose imports to handle paths, files etc
import glob
import pandas as pd
import numpy as np
import json
import subprocess

In [ ]:
%gui wx

In [ ]:
################
# Start FSLeyes 
################
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

In [ ]:
### Define Main Paths

base_root = repository_dir
deriv_root = op.join(base_root, 'derivatives')
preproc_root = op.join(base_root, 'derivatives','preprocessed_data')


### Create Necessary Folders for storing modified Images

mkdir_no_exist(deriv_root)
mkdir_no_exist(preproc_root)
mkdir_no_exist(op.join(preproc_root, 'anat'))
mkdir_no_exist(op.join(preproc_root, 'func'))
mkdir_no_exist(op.join(preproc_root, 'fmap'))

In [ ]:
### View T1 Image

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(base_root, 'T1','T1w.nii.gz'))

### Structural Preprocessing using fsl_anat(Skull-Stripping, Segmentation and Registration to MNI Space)

In [ ]:
### Define Paths

anatomical_path = op.join(base_root, 'T1', 'T1w.nii.gz')

### Run fsl_anat

import shutil

def fsl_anat_wrapped(anatomical_target, output_path):
    os.system('fsl_anat -i {} --clobber --nosubcortseg -o {}'.format(anatomical_target,output_path))
    # Now move all files from the output_path.anat folder created by FSL to 
    # the actual output_path
    fsl_anat_path = output_path+'.anat'
    files_to_move = glob.glob(op.join(fsl_anat_path, '*'))
    for f in files_to_move:
        shutil.move(f, op.join(output_path, op.split(f)[1]))
    
    # Remove the output_path.anat folder
    os.rmdir(fsl_anat_path)

fsl_anat_wrapped(anatomical_path, op.join(preproc_root, 'anat'))

In [ ]:
### View Mask Fit 

resulting_mask_path = op.join(preproc_root, 'anat', 'T1_biascorr_brain_mask.nii.gz')

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(base_root,'T1', 'T1w.nii.gz'))
fsleyesDisplay.load(resulting_mask_path)

In [ ]:
### View Betted Brain

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1_biascorr_brain.nii.gz'))
fsleyesDisplay.load(op.join(base_root,'T1', 'T1w.nii.gz'))

In [ ]:
### View Segmentation Results

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1_biascorr_brain.nii.gz'))
fsleyesDisplay.load(glob.glob(op.join(preproc_root, 'anat','*pve_0*'))[0])
fsleyesDisplay.load(glob.glob(op.join(preproc_root, 'anat','*pve_1*'))[0])
fsleyesDisplay.load(glob.glob(op.join(preproc_root, 'anat','*pve_2*'))[0])
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

In [ ]:
### Define Paths

betted_path_grey = op.join(preproc_root, 'anat', 'T1_biascorr_brain.nii.gz:colormap=greyscale')
mni_reference_path = op.expandvars('$FSLDIR/data/standard/MNI152_T1_1mm.nii.gz')

### View T1 alignment before Registration
import threading

def launch_freeview(img_list):
    """
    Wrapper around freeview to launch it with several images.
    This wrapper is necessary to launch freeview in a separate thread, ensuring the notebook is free to do something else.

    Parameters
    ----------
    img_list: list of string
        List of images (files) to load. Assumed by default to be volume files.
    """
    args = []
    
    for i in range(len(img_list)):
        args.append("-v")
        args.append(img_list[i])
    # Run the command
    subprocess.run(["freeview"] + args)

imgList = [mni_reference_path, 
           betted_path_grey] # Modify here this list to add any image you want, in .nii.gz format
freeview_thread = threading.Thread(target=launch_freeview, args=(imgList,)) # Remark the (imgList,) when passing to args. This is very important to make Thread work properly

# Start the thread
freeview_thread.start()

print("Freeview is running in a separate thread.")

In [ ]:
### View Fit Registration

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm')))
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1_to_MNI_lin'))
fsleyesDisplay.load(op.join(preproc_root, 'anat', 'T1_to_MNI_nonlin'))

# Functional Preprocessing

In [ ]:
### Define fMRI path

fMRI_path = op.join(base_root, 'fMRI', 'tfMRI_LANGUAGE_LR')

In [ ]:
### View fMRI data

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(op.join(fMRI_path, 'tfMRI_LANGUAGE_LR'))

In [ ]:
# View Scanning information
import nibabel as nib

img = nib.load(op.join(fMRI_path, 'tfMRI_LANGUAGE_LR.nii.gz'))
header = img.header


print("TR" , header.get_zooms()[3])

In [ ]:
### View mean voxel intensity to determine volumes to remove
import matplotlib.pyplot as plt
import nibabel as nib

data = nib.load(op.join(fMRI_path, 'tfMRI_LANGUAGE_LR.nii.gz')).get_fdata().mean(axis=(0,1,2))
print("fMRI EPI Length: ", len(data))

plt.plot(data[:20])
plt.xlabel('Time (volume)')
plt.ylabel('Mean voxel intensity')

### Don't remove volumes. The mean doesn't vary that much compared to what was seen in the lab. Also the onset of the task would be displaced relative to t=0 for the fMRI images (<0). WE can also assume the data comes from newer fMRI models due to the source of the data

## Motion Correction 

In [ ]:
from fsl.wrappers import mcflirt
### Define paths

path_original_data = op.join(fMRI_path, 'tfMRI_LANGUAGE_LR')
path_moco_data = op.join(preproc_root, 'func', 'tfMRI_LANGUAGE_LR_moco')

### Select Midline

mcflirt(infile=path_original_data,o=path_moco_data, plots=True, report=True, dof=6, mats=True)

In [ ]:
### Compute frame-wise displacement (FD)

def load_mot_params_fsl_6_dof(path):
    return pd.read_csv(path, sep='  ', header=None, 
            engine='python', names=['Rotation x', 'Rotation y', 'Rotation z','Translation x', 'Translation y', 'Translation z'])

def compute_FD_power(mot_params):
    framewise_diff = mot_params.diff().iloc[1:]

    rot_params = framewise_diff[['Rotation x', 'Rotation y', 'Rotation z']]
    # Estimating displacement on a 50mm radius sphere =: brain
    converted_rots = rot_params*50 
    trans_params = framewise_diff[['Translation x', 'Translation y', 'Translation z']]
    fd = converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)
    return fd

mot_params = load_mot_params_fsl_6_dof(op.join(preproc_root, 'func', 'tfMRI_LANGUAGE_LR_moco.par'))
fd = compute_FD_power(mot_params).to_numpy()

### Define mcflirt used threshold

threshold = np.quantile(fd,0.75) + 1.5*(np.quantile(fd,0.75) - np.quantile(fd,0.25))

### FD Plot

plt.plot(list(range(1, fd.size+1)), fd)
plt.title("Frame-wise Displacement")
plt.xlabel('Volume')
plt.ylabel('FD (mm)')
plt.hlines(threshold, 0, len(data),colors='black', linestyles='dashed', label='FD threshold') # len(data) from mean voxel intensity test
plt.legend()

image_root = op.join(base_root, 'Images')
plt.savefig(op.join(image_root, "FD"))

plt.show()



In [ ]:
print(f"Indices of frames above threshold: {(np.where(fd > threshold)[0] + 1).tolist()}")

In [ ]:
### View Motion Corrected EPI 

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(path_original_data)
fsleyesDisplay.load(path_moco_data)

## Coregistration func to anat

In [ ]:
from fsl.wrappers import epi_reg

### Define paths

epi_source = path_moco_data
whole_t1 = op.join(preproc_root, 'anat', 'T1_biascorr.nii.gz') #Use fsl_anat whole_t1 head, as it shares the same headers
skull_stripped_t1 = op.join(preproc_root, 'anat', 'T1_biascorr_brain.nii.gz')
output_path = op.join(preproc_root, 'func', 'tfMRI_LANGUAGE_LR_bbr')
ref_vol_name =  op.join(preproc_root, 'func', 'tfMRI_LANGUAGE_LR_vol_middle')
white_matter_segmentation = op.join(preproc_root, 'anat', 'T1_fast_pve_2.nii.gz')

### Extract middle volume

fslroi(epi_source, ref_vol_name, str(158), str(1))

### Run epi-reg using fast white matter segmentation 


subprocess.run(['epi_reg','--epi={}'.format(ref_vol_name), '--t1={}'.format(whole_t1), '--t1brain={}'.format(skull_stripped_t1), 
                '--out={}'.format(output_path),
               '--wmseg={}'.format(white_matter_segmentation)])


In [ ]:
### View Coregistration Result

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(skull_stripped_t1)
fsleyesDisplay.load(output_path)

In [ ]:
### Cleanup

def cleanup_epi_reg(path_to_clean):
    patterns = ['*_fast_*', '*_fieldmap*']
    for p in patterns:
        files = glob.glob(op.join(path_to_clean, p))
        for f in files:
            os.remove(f)

cleanup_epi_reg(op.join(preproc_root, 'func'))

## Smoothing

In [ ]:
### Define path
input_path = op.join(fMRI_path, 'tfMRI_LANGUAGE_LR.nii.gz')
output_path = op.join(op.join(preproc_root, 'func', 'tfMRI_LANGUAGE_LR'))

### Voxel Size 

voxel_size = nib.load(input_path).header.get_zooms()[:3]
print(voxel_size)

In [ ]:
FWHM = voxel_size[0] * 2

### Smoothing (4mm)
subprocess.run(['fslmaths',input_path, '-s', str(FWHM/2.3548), '{}_smoothed-4mm'.format(output_path)])

In [ ]:
### View Coregistration Result

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(input_path)
fsleyesDisplay.load(output_path + '_smoothed-4mm')

In [ ]:
%%writefile .gitignore

T1/

fMRI/

derivatives/

.ipynb_checkpoints/

__pycache__/